# JAX_wildlife.example
End-to-end workflow

In [ ]:
import logging, os
import matplotlib.pyplot as plt
from JAX_wildlife_utils import (
    load_dataset, TrainConfig, train, evaluate, plot_confusion_matrix, sample_misclassifications
)

logging.basicConfig(level=logging.INFO)
DATA_DIR = './data/animals10'
IMAGE_SIZE = (128, 128)
OUTPUT_DIR = 'outputs'
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [ ]:
Xs, ys, class_names = load_dataset(DATA_DIR, image_size=IMAGE_SIZE, splits=(0.7, 0.15, 0.15))
len(class_names), {split: arr.shape for split, arr in Xs.items()}

In [ ]:
config = TrainConfig(image_size=IMAGE_SIZE, num_classes=len(class_names), num_epochs=3, batch_size=64)
state, history = train(Xs['train'], ys['train'], Xs['val'], ys['val'], config)
history

In [ ]:
metrics = evaluate(state, Xs['test'], ys['test'], class_names)
fig = plot_confusion_matrix(metrics['confusion_matrix'], class_names)
fig.savefig(f'{OUTPUT_DIR}/confusion_matrix.png', bbox_inches='tight')
fig

In [ ]:
images, y_true, y_pred = sample_misclassifications(Xs['test'], ys['test'], metrics['y_pred'], k=6)
if len(images) > 0:
    fig, axes = plt.subplots(1, len(images), figsize=(3 * len(images), 3))
    if len(images) == 1:
        axes = [axes]
    for ax, img, yt, yp in zip(axes, images, y_true, y_pred):
        ax.imshow(img)
        ax.axis('off')
        ax.set_title(f'T:{class_names[yt]} / P:{class_names[yp]}', fontsize=8)
    fig.tight_layout()
    fig.savefig(f'{OUTPUT_DIR}/misclassifications.png', bbox_inches='tight')
    fig
else:
    print('No misclassifications in sampled subset.')